# Clinical NLP Pipeline — End-to-End Demo

**Predicting 30-Day Hospital Readmission from Discharge Summaries**

This notebook runs the full pipeline:
1. **Data Generation & Loading** — Synthetic MIMIC-IV-style clinical data
2. **Preprocessing** — Clinical text cleaning, tokenization, phrase detection
3. **Topic Modeling** — LDA topic discovery on discharge notes
4. **Feature Engineering** — TF-IDF, topic distributions, structured features, text statistics
5. **Prediction** — Logistic Regression, Random Forest, XGBoost, LightGBM
6. **Fairness Audit** — Disparity analysis across gender, insurance, age group
7. **Visualization** — Publication-quality figures

In [ ]:
import sys, os, logging, warnings
from pathlib import Path

# Project root setup
PROJECT_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Logging & warnings
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import yaml
import numpy as np
import pandas as pd

with open("config/config.yaml") as f:
    config = yaml.safe_load(f)

print(f"Project root: {PROJECT_ROOT}")
print("Config loaded successfully.")

## 1. Data Generation & Loading

Generate synthetic clinical data (patients, admissions, discharge notes) mimicking the MIMIC-IV schema, then load and merge into a single analysis-ready DataFrame.

In [ ]:
from src.generate_synthetic_data import run as generate_data
from src.data_loader import load_all, get_data_summary

# Generate synthetic data (200 patients -> ~300+ notes)
generate_data(output_dir="data", n_patients=200)

# Load & merge all tables
df = load_all(config, use_synthetic=True)
summary = get_data_summary(df)

print(f"\nDataset shape: {df.shape}")
df.head(3)

## 2. Preprocessing

Clean clinical text (remove PHI patterns, dates, special characters), tokenize with NLTK, and prepare documents for topic modeling and feature extraction.

In [ ]:
from src.preprocess import build_preprocessing_pipeline

processed = build_preprocessing_pipeline(df, config=config, use_scispacy=False, use_phrases=False)

print(f"Processed shape: {processed.shape}")
print(f"Avg tokens per note: {processed['num_tokens'].mean():.1f}")
print(f"\nSample cleaned text (first 200 chars):")
print(processed["cleaned_text"].iloc[0][:200])
print(f"\nSample tokens: {processed['tokens'].iloc[0][:15]}")

## 3. Topic Modeling (LDA)

Discover latent clinical topics in discharge notes using Latent Dirichlet Allocation. We grid-search over topic counts and select the best by coherence score.

In [ ]:
from src.topic_model import run_lda_pipeline

# Filter to eligible notes (readmission label >= 0)
eligible_mask = processed["readmission_30day"] >= 0
eligible_df = processed[eligible_mask].reset_index(drop=True)
tokenized_docs = eligible_df["tokens"].apply(lambda t: t if isinstance(t, list) else []).tolist()

print(f"Eligible documents for topic modeling: {len(tokenized_docs)}")

# Run LDA pipeline (grid search over num_topics)
lda_results = run_lda_pipeline(tokenized_docs, config=config)

print(f"\nBest number of topics: {lda_results['best_num_topics']}")
print(f"Coherence scores: {lda_results['coherence_scores']}")
print(f"\nTopic labels:")
for tid, label in lda_results["topic_labels"].items():
    words = ", ".join(w for w, _ in lda_results["topic_words"][tid][:6])
    print(f"  Topic {tid:2d} [{label}]: {words} ...")

In [ ]:
# Topic summary table
lda_results["topics_df"]

## 4. Feature Engineering

Build multiple feature representations:
- **TF-IDF** — Bag-of-words text representation
- **Topic distributions** — LDA document-topic probabilities
- **Structured** — Demographics, admission info, prior utilization
- **Text statistics** — Note length, negations, section count
- **Combined** — All of the above stacked

In [ ]:
from src.feature_engineer import build_feature_sets

feature_sets = build_feature_sets(processed, lda_results=lda_results, config=config)

print("Feature sets built:")
for key in ["tfidf", "topic_lda", "structured", "text_stats", "combined"]:
    entry = feature_sets.get(key)
    if entry and entry.get("X") is not None:
        shape = entry["X"].shape
        print(f"  {key:20s} — shape: {shape}")

print(f"\nLabels: {len(feature_sets['label'])} samples")
print(f"  Readmitted: {feature_sets['label'].sum()} ({feature_sets['label'].mean()*100:.1f}%)")

## 5. Prediction Pipeline

Train Logistic Regression, Random Forest, XGBoost, and LightGBM on each feature set. Evaluate with ROC-AUC, F1, PR-AUC, and cross-validation.

In [ ]:
from src.predict import run_prediction_pipeline

prediction_results = run_prediction_pipeline(feature_sets, config=config)

# Summary table
display_cols = ["model", "feature_type", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
results_df = prediction_results["results_df"]
results_df[display_cols].sort_values("roc_auc", ascending=False)

In [ ]:
# Best model summary
best = prediction_results["best"]
print(f"Best model: {best.get('model')} + {best.get('feature_type')}")
print(f"  ROC-AUC: {best.get('roc_auc', 0):.4f}")
print(f"  F1:      {best.get('f1', 0):.4f}")
print(f"  PR-AUC:  {best.get('pr_auc', 0):.4f}")

# Cross-validation results
if prediction_results["cv_results"]:
    print("\nCross-validation results:")
    for cv in prediction_results["cv_results"]:
        print(f"  {cv['model']} + {cv['feature_type']} — "
              f"F1: {cv['f1_mean']:.3f}±{cv['f1_std']:.3f} | "
              f"ROC-AUC: {cv['roc_auc_mean']:.3f}±{cv['roc_auc_std']:.3f}")

## 6. Fairness Audit

Evaluate prediction fairness across protected attributes (gender, insurance, age group) using Fairlearn. We check for demographic parity, equalized odds, and false positive/negative rate disparities.

In [ ]:
from src.fairness import run_fairness_audit

# Use the best model's predictions for the fairness audit
best_model_name = best.get("model", "logistic_regression")
best_feat_type = best.get("feature_type", "structured")
best_model = prediction_results["models"].get((best_model_name, best_feat_type))
best_splits = prediction_results["splits"].get(best_feat_type)

if best_model and best_splits:
    y_test = best_splits["y_test"]
    X_test = best_splits["X_test"]
    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # Get the eligible DataFrame aligned with the test split
    eligible_df = feature_sets["eligible_df"]

    # Reconstruct test indices via the split
    from src.predict import split_data
    test_size = config.get("prediction", {}).get("test_size", 0.2)
    val_size = config.get("prediction", {}).get("val_size", 0.1)
    feat_entry = feature_sets.get(
        {"tfidf": "tfidf", "topic_distribution": "topic_lda",
         "structured": "structured", "combined": "combined"}.get(best_feat_type, best_feat_type)
    )
    X_full = feat_entry["X"]
    labels = feature_sets["label"]

    from sklearn.model_selection import train_test_split
    _, X_test_idx, _, _ = train_test_split(
        np.arange(len(labels)), labels,
        test_size=test_size, stratify=labels, random_state=42,
    )

    # Build protected attributes DataFrame for test set
    protected_cols = config.get("fairness", {}).get("protected_attributes", ["gender", "insurance", "age_group"])
    protected_df = eligible_df.loc[X_test_idx, protected_cols].reset_index(drop=True)

    fairness_results = run_fairness_audit(
        y_true=y_test,
        y_pred=y_pred,
        y_prob=y_prob,
        protected_df=protected_df,
        config=config,
    )
else:
    print("Best model not available for fairness audit.")

In [ ]:
# Fairness summary
if fairness_results:
    fairness_results["summary_df"]

## 7. Visualization

Generate all publication-quality figures and save to `results/figures/`.

In [ ]:
from src.visualize import generate_all_figures

# Add readmission labels to lda_results for the heatmap
lda_results["readmission_labels"] = feature_sets["label"]

saved_figures = generate_all_figures(
    merged_df=df,
    prediction_results=prediction_results,
    lda_results=lda_results,
    fairness_results=fairness_results if "fairness_results" in dir() else None,
    config=config,
)

print(f"\nGenerated {len(saved_figures)} figures:")
for fig_path in saved_figures:
    print(f"  {fig_path}")

In [ ]:
# Display key figures inline
from IPython.display import Image, display

key_figures = ["demographics", "roc_curves", "model_comparison", "fairness_disparity"]
for name in key_figures:
    fig_path = f"results/figures/{name}.png"
    if Path(fig_path).exists():
        print(f"\n{'='*60}")
        print(f"  {name.replace('_', ' ').title()}")
        print(f"{'='*60}")
        display(Image(filename=fig_path, width=800))

## Summary

| Component | Details |
|-----------|---------|
| **Data** | Synthetic MIMIC-IV (200 patients, ~300+ notes) |
| **Preprocessing** | PHI removal, NLTK tokenization, clinical stopwords |
| **Topic Modeling** | LDA with coherence-based topic selection |
| **Features** | TF-IDF, topic distributions, structured, text stats, combined |
| **Models** | Logistic Regression, Random Forest, XGBoost, LightGBM |
| **Fairness** | Demographic parity, equalized odds, FPR/FNR disparity |
| **Output** | Publication-quality figures in `results/figures/` |